# 🏁 Chess Move Prediction with AlphaZero‑Style Network
**Regularized Residual CNN**  
Train a compact policy network on PGN games of a target player, then play against the model.

---
### Overview
- Input: 19‑plane board tensor (piece positions, side to move, castling rights, etc.)
- Output: 4672‑class policy (73 planes × 64 squares) representing move probabilities
- Training: cross‑entropy with label smoothing, weight decay, dropout, and game‑level data split to avoid leakage
- Inference: temperature‑adjusted sampling over legal moves

Designed to stop overfitting on small datasets (~350k parameters).

In [2]:
# ========================
# IMPORTS
# ========================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import chess.pgn
import chess
import numpy as np
import random
import os

print('Libraries imported successfully.')

Libraries imported successfully.


## ⚙️ Configuration
Adjust these parameters to match your hardware and dataset.

In [12]:
# ========================
# CONFIGURATION
# ========================
PGN_FILE = "D:\\mohammad\\Programming\\Pytthon_4_AI\\NTI_DL\\Fischer.pgn"               # Your PGN file
PLAYER_NAME = "Fischer, R"             # Exact name from headers (White/Black field)
BATCH_SIZE = 512                       # Default for GPU (auto‑scaled for CPU)
EPOCHS = 50
LEARNING_RATE = 1e-3
LABEL_SMOOTHING = 0.1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

Using device: cpu


## 🧠 Feature Engineering
### 19‑plane board tensor
Encodes the full position: 12 planes for pieces (6 white + 6 black), 7 auxiliary planes.

In [4]:
def board_to_tensor(board: chess.Board) -> torch.Tensor:
    """Convert a python‑chess board to a (19,8,8) float tensor."""
    piece_map = {chess.PAWN:0, chess.KNIGHT:1, chess.BISHOP:2,
                 chess.ROOK:3, chess.QUEEN:4, chess.KING:5}
    tensor = torch.zeros(19, 8, 8, dtype=torch.float32)

    # Pieces (0-11)
    for sq, p in board.piece_map().items():
        row, col = divmod(sq, 8)
        ch = piece_map[p.piece_type] if p.color == chess.WHITE else 6+piece_map[p.piece_type]
        tensor[ch, row, col] = 1.0

    # Side to move (12)
    if board.turn == chess.WHITE:
        tensor[12,:,:] = 1.0

    # En passant square (13)
    if board.ep_square is not None:
        r, c = divmod(board.ep_square, 8)
        tensor[13, r, c] = 1.0

    # Castling rights (14-17)
    cr = board.castling_rights
    tensor[14,:,:] = 1.0 if cr & chess.BB_H1 else 0.0
    tensor[15,:,:] = 1.0 if cr & chess.BB_A1 else 0.0
    tensor[16,:,:] = 1.0 if cr & chess.BB_H8 else 0.0
    tensor[17,:,:] = 1.0 if cr & chess.BB_A8 else 0.0

    # Fifty‑move counter (18)
    tensor[18,:,:] = board.halfmove_clock / 100.0

    return tensor

### Move → policy index
Maps a legal chess move to a flat index (0‑4671) following the AlphaZero 73‑plane encoding.

In [5]:
def move_to_policy_index(move: chess.Move):
    """
    Returns (plane, row, col) or None if the move cannot be encoded.
    Flat index = plane * 64 + row * 8 + col
    """
    fr, fc = divmod(move.from_square, 8)
    tr, tc = divmod(move.to_square, 8)
    dr, dc = tr - fr, tc - fc

    # Knight moves (planes 56‑63)
    knight_offsets = [(-2,-1), (-2,1), (-1,-2), (-1,2), (1,-2), (1,2), (2,-1), (2,1)]
    if (abs(dr), abs(dc)) in [(1,2), (2,1)]:
        try:
            idx = knight_offsets.index((dr, dc))
            return 56 + idx, tr, tc
        except ValueError:
            return None

    # Queen‑like moves (planes 0‑55, 8 directions × 7 distances)
    dirs = [(-1,0), (-1,1), (0,1), (1,1), (1,0), (1,-1), (0,-1), (-1,-1)]
    for dir_idx, (ddr, ddc) in enumerate(dirs):
        if (ddr == 0 and ddc == 0) or (dr == 0 and dc == 0):
            continue
        if ddr != 0 and ddc != 0:  # diagonal
            if abs(dr) != abs(dc): continue
            if dr//abs(dr) != ddr or dc//abs(dc) != ddc: continue
            dist = abs(dr)
        elif ddr == 0:  # horizontal
            if dr != 0: continue
            if dc//abs(dc) != ddc: continue
            dist = abs(dc)
        else:  # vertical
            if dc != 0: continue
            if dr//abs(dr) != ddr: continue
            dist = abs(dr)
        if 1 <= dist <= 7:
            plane = dir_idx * 7 + (dist - 1)
            return plane, tr, tc

    # Underpromotions (planes 64‑72)
    if move.promotion and move.promotion != chess.QUEEN:
        if tr == 7:        # White
            forward_dr, left_dc, right_dc = -1, -1, 1
        elif tr == 0:      # Black
            forward_dr, left_dc, right_dc = 1, 1, -1
        else:
            return None

        if dr == forward_dr and dc == 0:
            dir_idx = 1   # forward
        elif dr == forward_dr and dc == left_dc:
            dir_idx = 0   # left
        elif dr == forward_dr and dc == right_dc:
            dir_idx = 2   # right
        else:
            return None

        if move.promotion == chess.KNIGHT:
            piece_offset = 0
        elif move.promotion == chess.BISHOP:
            piece_offset = 3
        elif move.promotion == chess.ROOK:
            piece_offset = 6
        else:
            return None

        plane = 64 + piece_offset + dir_idx
        return plane, tr, tc

    return None

## 📂 Dataset – Game‑level split to prevent leakage
Positions from the **same game** are never split between training and validation.

In [6]:
class ChessDataset(Dataset):
    """Torch dataset for (board_tensor, policy_target) pairs."""
    def __init__(self, positions, policy_targets):
        self.positions = positions
        self.policy_targets = policy_targets

    def __len__(self):
        return len(self.positions)

    def __getitem__(self, idx):
        return self.positions[idx], torch.tensor(self.policy_targets[idx], dtype=torch.long)


def parse_pgn_to_games(file_path, player_name):
    """Extract games where `player_name` appears as White or Black."""
    games_list = []
    with open(file_path, encoding='utf-8', errors='ignore') as f:
        while True:
            game = chess.pgn.read_game(f)
            if game is None:
                break
            white = game.headers.get("White", "")
            black = game.headers.get("Black", "")

            if player_name in white:
                games_list.append((game, chess.WHITE))
            elif player_name in black:
                games_list.append((game, chess.BLACK))
    return games_list


def games_to_features(games_subset):
    """Convert list of (game, target_color) to tensor features and flat policy targets."""
    positions, targets = [], []
    for game, target_color in games_subset:
        board = game.board()
        for move in game.mainline_moves():
            if board.turn == target_color:
                tensor = board_to_tensor(board)
                idx = move_to_policy_index(move)
                if idx is not None:
                    plane, r, c = idx
                    flat = plane * 64 + r * 8 + c
                    positions.append(tensor)
                    targets.append(flat)
            board.push(move)
    return positions, targets

## 🏗️ Model – Compact Residual CNN
**Regularization techniques:**  
- Reduced channel count (64 instead of 128)  
- Weight decay (AdamW)  
- Label smoothing  
- Dropout in the policy head  
- ~350k parameters to fight overfitting on small datasets

In [7]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        x = torch.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x = torch.relu(x + residual)
        return x


class ChessNet(nn.Module):
    def __init__(self, input_channels=19, num_blocks=4):
        super().__init__()
        self.conv_input = nn.Sequential(
            nn.Conv2d(input_channels, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        self.res_blocks = nn.Sequential(*[ResidualBlock(64) for _ in range(num_blocks)])
        
        self.policy_head = nn.Sequential(
            nn.Conv2d(64, 73, kernel_size=1, bias=False),
            nn.BatchNorm2d(73),
            nn.ReLU(),
            nn.Flatten(),
            nn.Dropout(0.3)   # 30% drop during training
        )

    def forward(self, x):
        x = self.conv_input(x)
        x = self.res_blocks(x)
        policy = self.policy_head(x)
        return policy


# Quick sanity check
dummy = ChessNet()
print(f"Total parameters: {sum(p.numel() for p in dummy.parameters()):,}")

Total parameters: 311,826


## 🎯 Training loop
Includes automatic mixed precision (AMP) for GPUs and cosine annealing schedule.

In [8]:
def train_model(model, train_loader, val_loader, epochs, lr, device):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    use_amp = (device.type == 'cuda')
    scaler = torch.amp.GradScaler('cuda') if use_amp else None

    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0
        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            if use_amp:
                with torch.amp.autocast('cuda'):
                    logits = model(x)
                    loss = criterion(logits, y)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(x)
                loss = criterion(logits, y)
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * x.size(0)
            _, pred = torch.max(logits, 1)
            correct += (pred == y).sum().item()
            total += y.size(0)

        train_acc = correct / total
        scheduler.step()

        # Validation
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)
                logits = model(x)
                _, pred = torch.max(logits, 1)
                val_correct += (pred == y).sum().item()
                val_total += y.size(0)
        val_acc = val_correct / val_total
        print(f"Epoch {epoch+1:02d}/{epochs} | Loss: {total_loss/total:.4f} | "
              f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

## ♟️ Inference & gameplay
`pick_move` samples from the softmax‑output with temperature control.

In [9]:
def pick_move(model, board, temperature=0.5, device='cpu'):
    model.eval()
    with torch.no_grad():
        tensor = board_to_tensor(board).unsqueeze(0).to(device)
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1).squeeze(0).cpu().numpy()

    legal_moves = list(board.legal_moves)
    if not legal_moves:
        return random.choice(list(board.legal_moves)) if len(board.legal_moves) > 0 else None

    move_probs = []
    for move in legal_moves:
        idx = move_to_policy_index(move)
        if idx is not None:
            plane, r, c = idx
            flat_idx = plane * 64 + r * 8 + c
            move_probs.append(probs[flat_idx])
        else:
            move_probs.append(0.0)

    move_probs = np.array(move_probs)
    total = move_probs.sum()
    if total > 0:
        move_probs /= total
    else:
        return random.choice(legal_moves)

    if temperature > 0:
        move_probs = np.exp(np.log(move_probs + 1e-9) / temperature)
        move_probs /= move_probs.sum()
        return np.random.choice(legal_moves, p=move_probs)
    else:
        return legal_moves[np.argmax(move_probs)]


def play_vs_human(model, device='cpu', model_plays_white=True):
    board = chess.Board()
    while not board.is_game_over(claim_draw=True):
        print("\n", board)
        if (board.turn == chess.WHITE) == model_plays_white:
            move = pick_move(model, board, temperature=0.1, device=device)
            print(f"Model plays: {move}")
            board.push(move)
        else:
            while True:
                uci = input("Your move (UCI): ")
                try:
                    board.push_uci(uci)
                    break
                except ValueError:
                    print("Illegal move.")
    print("\nGame over:", board.result())

## 💾 Save / Load helpers

In [10]:
def save_model(model, path="chess_resnet.pth"):
    # Handle torch.compile wrapper if present
    state_dict = model._orig_mod.state_dict() if hasattr(model, '_orig_mod') else model.state_dict()
    torch.save(state_dict, path)
    print(f"Model saved to {path}")

def load_model(model, path="chess_resnet.pth", device='cpu'):
    model.load_state_dict(torch.load(path, map_location=device))
    model.eval()
    print(f"Model loaded from {path}")
    return model

## 🚀 Main Execution
1. Parse PGN and extract game‑level splits  
2. Create data loaders (auto‑detects CPU/GPU capability)  
3. Train the model  
4. Save it  
5. Interactive game against the model

In [ ]:
# ========================
# MAIN PIPELINE
# ========================
print("Parsing PGN into unique games...")
all_games = parse_pgn_to_games(PGN_FILE, PLAYER_NAME)
print(f"Total structured matches identified: {len(all_games)}")

if len(all_games) == 0:
    print("No matches parsed – check PGN file name or player metadata strings.")
else:
    # Shuffle and split games (80% train, 20% val)
    random.seed(42)
    random.shuffle(all_games)
    split_idx = int(0.8 * len(all_games))
    train_games = all_games[:split_idx]
    val_games = all_games[split_idx:]

    print("Extracting features from training game array...")
    train_pos, train_tar = games_to_features(train_games)
    print(f"Training states extracted: {len(train_pos)}")

    print("Extracting features from validation game array...")
    val_pos, val_tar = games_to_features(val_games)
    print(f"Validation states extracted: {len(val_pos)}")

    # Create datasets
    train_set = ChessDataset(train_pos, train_tar)
    val_set = ChessDataset(val_pos, val_tar)

    # System accelerator checks
    cuda_is_working = torch.cuda.is_available() and (torch.cuda.get_device_capability(0)[0] >= 3)
    if not cuda_is_working:
        print("\n⚠️ Running via execution thread: LOCAL HOST CPU")
        DEVICE = torch.device("cpu")
        CURRENT_BATCH_SIZE = 128   # safe for system RAM
        NUM_WORKERS = 0
        PIN_MEMORY = False
    else:
        print("\n🚀 Running via execution thread: GPU STACK")
        DEVICE = torch.device("cuda")
        CURRENT_BATCH_SIZE = BATCH_SIZE
        NUM_WORKERS = 2
        PIN_MEMORY = True

    train_loader = DataLoader(train_set, batch_size=CURRENT_BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    val_loader = DataLoader(val_set, batch_size=CURRENT_BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    # Build model
    model = ChessNet(input_channels=19, num_blocks=4)
    print(f"Adjusted Parameters: {sum(p.numel() for p in model.parameters()):,}")

    if DEVICE.type == 'cuda':
        model = torch.compile(model)  # PyTorch 2.0+ optimization

    # Train
    train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LEARNING_RATE, device=DEVICE)

    # Save
    save_model(model)

    # Play a game (uncomment when ready)
    print("\nPlay a game! (Model as White)")
    play_vs_human(model, device=DEVICE, model_plays_white=True)

Parsing PGN into unique games...
Total structured matches identified: 827
Extracting features from training game array...
Training states extracted: 27049
Extracting features from validation game array...
Validation states extracted: 6709

⚠️ Running via execution thread: LOCAL HOST CPU
Adjusted Parameters: 311,826
Epoch 01/50 | Loss: 6.7876 | Train Acc: 0.1210 | Val Acc: 0.2076
Epoch 02/50 | Loss: 5.8226 | Train Acc: 0.1999 | Val Acc: 0.2488
Epoch 03/50 | Loss: 5.5377 | Train Acc: 0.2336 | Val Acc: 0.2470
Epoch 04/50 | Loss: 5.4154 | Train Acc: 0.2575 | Val Acc: 0.2638
Epoch 05/50 | Loss: 5.2722 | Train Acc: 0.2774 | Val Acc: 0.2641
Epoch 06/50 | Loss: 5.0875 | Train Acc: 0.3103 | Val Acc: 0.2652
Epoch 07/50 | Loss: 5.0238 | Train Acc: 0.3330 | Val Acc: 0.2658
Epoch 08/50 | Loss: 4.8908 | Train Acc: 0.3587 | Val Acc: 0.2668
Epoch 09/50 | Loss: 4.7848 | Train Acc: 0.3866 | Val Acc: 0.2701
Epoch 10/50 | Loss: 4.7296 | Train Acc: 0.4031 | Val Acc: 0.2789
Epoch 11/50 | Loss: 4.6196 | Trai